In [1]:

import pyreadr
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression, LassoCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from scipy.optimize import nnls    # for short-stacking weights
import warnings
import os
warnings.filterwarnings('ignore')

# Helper: short-stacking weights via NNLS + renormalization to sum to 1.
# Matches the convention used by R Code/ExampleMWDiD.R and Stata
# pystacked/ddml (Python NNLS path -- see PLAN.md decision
# "Constrained stacking QP in Python: NNLS + renormalize").
def nnls_normalize(target, P):
    target = np.asarray(target, dtype=float).ravel()
    P = np.asarray(P, dtype=float)
    try:
        w, _ = nnls(P, target)
    except Exception:
        w = np.full(P.shape[1], 1.0 / P.shape[1])
    s = w.sum()
    if s > 0:
        w = w / s
    else:
        w = np.full(P.shape[1], 1.0 / P.shape[1])
    return w

# Set relative path from Python code/ to Data/
rdata_path = os.path.join("..", "Data", "mw_data_ch.RData")

# Load the RData file
result = pyreadr.read_r(rdata_path)

mw_data_ch = list(result.values())[0]  # fallback in case the key is unknown

# Save it as a CSV file inside Data/
mw_data_ch.to_csv("../Data/mw_data_ch.csv", index=False)

# Later, to load the CSV file again:
csv_path = os.path.join("..", "Data", "mw_data_ch.csv")
data = pd.read_csv(csv_path)


In [2]:


# Set random seed for reproducibility
np.random.seed(772023)

# Load data (assumes you have exported mw_data_ch.RData to CSV)
data = pd.read_csv("../Data/mw_data_ch.csv")

# Filter rows: keep only those with emp0A01_BS > 0 and annual_avg_pay > 0
data = data[data['emp0A01_BS'] > 0]
data = data[data['annual_avg_pay'] > 0]

# Drop rows with missing values except for 'state_mw' and 'fed_mw'
cols_to_check = [col for col in data.columns if col not in ['state_mw', 'fed_mw']]
data = data.dropna(subset=cols_to_check)

# Create new columns: log employment, log population, log average pay
data['lemp'] = np.log(data['emp0A01_BS'])
data['pop'] = pd.to_numeric(data['pop'], errors='coerce')
data['lpop'] = np.log(data['pop'])
data['lavg_pay'] = np.log(data['annual_avg_pay'])

# Create region variable as in R
def region_func(row):
    if row['censusdiv'] in [1, 2]:
        return 1
    elif row['censusdiv'] in [3, 4]:
        return 2
    elif row['censusdiv'] in [5, 6, 7]:
        return 3
    elif row['censusdiv'] in [8, 9]:
        return 4
    else:
        return np.nan

data['region'] = data.apply(region_func, axis=1).astype('category')

# Create ever_treated and id columns
data['ever_treated'] = (data['G'] != 0).astype(int)
data['id'] = data['countyreal']

# Drop already and early-treated
data = data[(data['G'] == 0) | (data['G'] > 2001)]

In [3]:
# Drop all the variables we won't use
drop_cols = [
    "countyreal", "state_name", "FIPS", "emp0A01_BS", "quarter",
    "censusdiv", "pop", "annual_avg_pay", "state_mw", "fed_mw",
    "ever_treated"
]
data = data.drop(columns=drop_cols)

# Create treatment and control subsets for each year
treat1 = data[(data['G'] == 2004) & (data['year'] == 2001)].copy()
treat2 = data[(data['G'] == 2004) & (data['year'] == 2002)].copy()
treat3 = data[(data['G'] == 2004) & (data['year'] == 2003)].copy()
treat4 = data[(data['G'] == 2004) & (data['year'] == 2004)].copy()
treat5 = data[(data['G'] == 2004) & (data['year'] == 2005)].copy()
treat6 = data[(data['G'] == 2004) & (data['year'] == 2006)].copy()
treat7 = data[(data['G'] == 2004) & (data['year'] == 2007)].copy()

cont1 = data[((data['G'] == 0) | (data['G'] > 2001)) & (data['year'] == 2001)].copy()
cont2 = data[((data['G'] == 0) | (data['G'] > 2002)) & (data['year'] == 2002)].copy()
cont3 = data[((data['G'] == 0) | (data['G'] > 2003)) & (data['year'] == 2003)].copy()
cont4 = data[((data['G'] == 0) | (data['G'] > 2004)) & (data['year'] == 2004)].copy()
cont5 = data[((data['G'] == 0) | (data['G'] > 2005)) & (data['year'] == 2005)].copy()
cont6 = data[((data['G'] == 0) | (data['G'] > 2006)) & (data['year'] == 2006)].copy()
cont7 = data[((data['G'] == 0) | (data['G'] > 2007)) & (data['year'] == 2007)].copy()

In [4]:
# Remove columns for "control" variables
drop_vars = ["year", "G", "region", "treated"]
treat1 = treat1.drop(columns=[col for col in drop_vars if col in treat1.columns])
cont1 = cont1.drop(columns=[col for col in drop_vars if col in cont1.columns])

# 2003 will serve as pre period
treatB = pd.merge(treat3, treat1, on="id", suffixes=(".pre", ".0"))
treatB = treatB.drop(columns=[col for col in ["treated", "lpop.pre", "lavg_pay.pre", "year", "G"] if col in treatB.columns])

contB = pd.merge(cont3, cont1, on="id", suffixes=(".pre", ".0"))
contB = contB.drop(columns=[col for col in ["treated", "lpop.pre", "lavg_pay.pre", "year", "G"] if col in contB.columns])

# Remove columns for treatment and control sets for 2004-2007
drop_vars2 = ["lpop", "lavg_pay", "year", "G", "region"]
treat4 = treat4.drop(columns=[col for col in drop_vars2 if col in treat4.columns])
treat5 = treat5.drop(columns=[col for col in drop_vars2 if col in treat5.columns])
treat6 = treat6.drop(columns=[col for col in drop_vars2 if col in treat6.columns])
treat7 = treat7.drop(columns=[col for col in drop_vars2 if col in treat7.columns])
cont4 = cont4.drop(columns=[col for col in drop_vars2 if col in cont4.columns])
cont5 = cont5.drop(columns=[col for col in drop_vars2 if col in cont5.columns])
cont6 = cont6.drop(columns=[col for col in drop_vars2 if col in cont6.columns])
cont7 = cont7.drop(columns=[col for col in drop_vars2 if col in cont7.columns])

# Merge and create difference variables for treatment groups
tdid04 = pd.merge(treat4, treatB, on="id")
tdid04["dy"] = tdid04["lemp"] - tdid04["lemp.pre"]
tdid04 = tdid04.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in tdid04.columns])

tdid05 = pd.merge(treat5, treatB, on="id")
tdid05["dy"] = tdid05["lemp"] - tdid05["lemp.pre"]
tdid05 = tdid05.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in tdid05.columns])

tdid06 = pd.merge(treat6, treatB, on="id")
tdid06["dy"] = tdid06["lemp"] - tdid06["lemp.pre"]
tdid06 = tdid06.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in tdid06.columns])

tdid07 = pd.merge(treat7, treatB, on="id")
tdid07["dy"] = tdid07["lemp"] - tdid07["lemp.pre"]
tdid07 = tdid07.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in tdid07.columns])

# Merge and create difference variables for control groups
cdid04 = pd.merge(cont4, contB, on="id")
cdid04["dy"] = cdid04["lemp"] - cdid04["lemp.pre"]
cdid04 = cdid04.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in cdid04.columns])

cdid05 = pd.merge(cont5, contB, on="id")
cdid05["dy"] = cdid05["lemp"] - cdid05["lemp.pre"]
cdid05 = cdid05.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in cdid05.columns])

cdid06 = pd.merge(cont6, contB, on="id")
cdid06["dy"] = cdid06["lemp"] - cdid06["lemp.pre"]
cdid06 = cdid06.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in cdid06.columns])

cdid07 = pd.merge(cont7, contB, on="id")
cdid07["dy"] = cdid07["lemp"] - cdid07["lemp.pre"]
cdid07 = cdid07.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in cdid07.columns])

In [5]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression, LassoCV, RidgeCV, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.preprocessing import PolynomialFeatures

# Prepare lists of treatment and control dataframes for each year
treat_dfs = [tdid04, tdid05, tdid06, tdid07]
cont_dfs = [cdid04, cdid05, cdid06, cdid07]
years = [2004, 2005, 2006, 2007]

# Columns of att/se_att: 0-8 per-learner (No Controls, Basic, Expansion,
#                        Lasso, Ridge, RF, Deep Tree, Shallow Tree, Tree CV)
#                        9 Best (min-MSE per arm)
#                        10 Stack (NNLS short-stack, renormalized)
att = np.full((4, 11), np.nan)
se_att = np.full((4, 11), np.nan)
RMSE_d = np.full((4, 9), np.nan)
RMSE_y = np.full((4, 9), np.nan)
trimmed = np.full((4, 9), np.nan)
stack_w_y_post = np.full((4, 9), np.nan)
stack_w_d_post = np.full((4, 9), np.nan)
preds_post = []   # per-year cross-fit predictions, concatenated and saved at end

# Loop over each year
for ii in range(4):
    # Prepare data for this year
    tdata = treat_dfs[ii].copy()
    cdata = cont_dfs[ii].copy()
    tdata['treated'] = 1
    cdata['treated'] = 0
    usedata = pd.concat([tdata, cdata], ignore_index=True)

    n = usedata.shape[0]
    Kf = 5
    kf = KFold(n_splits=Kf, shuffle=True, random_state=772023)
    
    # Arrays to store cross-fitted predictions
    yGd0x_fit = np.zeros((n, 9))
    dGx_fit = np.zeros((n, 9))
    pd_fit = np.zeros((n, 1))

    # Prepare features for expansion
    features = ['region', 'lemp.0', 'lpop.0', 'lavg_pay.0']
    X = usedata[features].copy()
    X = pd.get_dummies(X, columns=['region'], drop_first=True)
    poly = PolynomialFeatures(degree=3, include_bias=False)
    X_poly = poly.fit_transform(X)

    # Cross-fitting loop
    for k, (train_idx, test_idx) in enumerate(kf.split(usedata)):
        ktrain = usedata.iloc[train_idx]
        ktest = usedata.iloc[test_idx]
        xtrain = X_poly[train_idx]
        xtest = X_poly[test_idx]
        ytrain = ktrain['dy'].values
        dtrain = ktrain['treated'].values

        # P(D=1)
        pd_fit[test_idx, 0] = dtrain.mean()

        # E[D|X] (propensity score models)
        # 1) Constant
        dGx_fit[test_idx, 0] = dtrain.mean()
        # 2) Baseline controls (logistic regression)
        logit = LogisticRegression(solver='lbfgs', max_iter=1000)
        logit.fit(ktrain[features], dtrain)
        dGx_fit[test_idx, 1] = logit.predict_proba(ktest[features])[:, 1]
        # 3) Region-specific linear index (logistic regression with expansion)
        logit2 = LogisticRegression(solver='lbfgs', max_iter=1000)
        logit2.fit(xtrain, dtrain)
        dGx_fit[test_idx, 2] = logit2.predict_proba(xtest)[:, 1]
        # 4) Lasso (CV)
        lasso = LassoCV(cv=5)
        lasso.fit(xtrain, dtrain)
        dGx_fit[test_idx, 3] = lasso.predict(xtest)
        # 5) Ridge (CV)
        ridge = RidgeCV(alphas=np.logspace(-6, 6, 13), cv=5)
        ridge.fit(xtrain, dtrain)
        dGx_fit[test_idx, 4] = ridge.predict(xtest)
        # 6) Random forest
        rf = RandomForestClassifier(n_estimators=1000, max_features=4, random_state=772023)
        rf.fit(ktrain[features], dtrain)
        dGx_fit[test_idx, 5] = rf.predict_proba(ktest[features])[:, 1]
        # 7) Deep tree
        dt_deep = DecisionTreeClassifier(max_depth=15, min_samples_split=10, random_state=772023)
        dt_deep.fit(ktrain[features], dtrain)
        dGx_fit[test_idx, 6] = dt_deep.predict_proba(ktest[features])[:, 1]
        # 8) Shallow tree
        dt_shallow = DecisionTreeClassifier(max_depth=3, min_samples_split=10, random_state=772023)
        dt_shallow.fit(ktrain[features], dtrain)
        dGx_fit[test_idx, 7] = dt_shallow.predict_proba(ktest[features])[:, 1]
        # 9) Tree (CV) - use deep tree as a proxy
        dGx_fit[test_idx, 8] = dt_deep.predict_proba(ktest[features])[:, 1]

        # E[Y|D=0,X] (outcome models for controls)
        ktrain0 = ktrain[ktrain['treated'] == 0]
        xtrain0 = X_poly[train_idx][ktrain['treated'] == 0]
        ytrain0 = ktrain0['dy'].values

        # 1) Constant
        yGd0x_fit[test_idx, 0] = ytrain0.mean() if len(ytrain0) > 0 else 0
        # 2) Baseline controls (linear regression)
        lm = LinearRegression()
        lm.fit(ktrain0[features], ytrain0)
        yGd0x_fit[test_idx, 1] = lm.predict(ktest[features])
        # 3) Region-specific linear index (linear regression with expansion)
        lm2 = LinearRegression()
        lm2.fit(xtrain0, ytrain0)
        yGd0x_fit[test_idx, 2] = lm2.predict(xtest)
        # 4) Lasso (CV)
        lasso_y = LassoCV(cv=5)
        lasso_y.fit(xtrain0, ytrain0)
        yGd0x_fit[test_idx, 3] = lasso_y.predict(xtest)
        # 5) Ridge (CV)
        ridge_y = RidgeCV(alphas=np.logspace(-6, 6, 13), cv=5)
        ridge_y.fit(xtrain0, ytrain0)
        yGd0x_fit[test_idx, 4] = ridge_y.predict(xtest)
        # 6) Random forest
        rf_y = RandomForestRegressor(n_estimators=1000, max_features=4, random_state=772023)
        rf_y.fit(ktrain0[features], ytrain0)
        yGd0x_fit[test_idx, 5] = rf_y.predict(ktest[features])
        # 7) Deep tree
        dt_deep_y = DecisionTreeRegressor(max_depth=15, min_samples_split=10, random_state=772023)
        dt_deep_y.fit(ktrain0[features], ytrain0)
        yGd0x_fit[test_idx, 6] = dt_deep_y.predict(ktest[features])
        # 8) Shallow tree
        dt_shallow_y = DecisionTreeRegressor(max_depth=3, min_samples_split=10, random_state=772023)
        dt_shallow_y.fit(ktrain0[features], ytrain0)
        yGd0x_fit[test_idx, 7] = dt_shallow_y.predict(ktest[features])
        # 9) Tree (CV) - use deep tree as a proxy
        yGd0x_fit[test_idx, 8] = dt_deep_y.predict(ktest[features])

    # Calculate RMSE for D and Y
    RMSE_d[ii, :] = np.sqrt(np.mean((usedata['treated'].values.reshape(-1, 1) - dGx_fit) ** 2, axis=0))
    RMSE_y[ii, :] = np.sqrt(np.mean((usedata['dy'].values[usedata['treated'] == 0].reshape(-1, 1) -
                                    yGd0x_fit[usedata['treated'] == 0, :]) ** 2, axis=0))

    # Short-stack on un-trimmed cross-fit predictions.
    # Y model fit on D==0 rows; D model fit on full sample.
    d0_mask = usedata['treated'].values == 0
    w_y = nnls_normalize(usedata['dy'].values[d0_mask],
                         yGd0x_fit[d0_mask, :])
    w_d = nnls_normalize(usedata['treated'].values, dGx_fit)
    yGd0_ss = yGd0x_fit @ w_y
    dGx_ss  = dGx_fit  @ w_d
    stack_w_y_post[ii, :] = w_y
    stack_w_d_post[ii, :] = w_d

    # Snapshot raw predictions for downstream diagnostics, then trim.
    model_id = "m%02d" % (4 + ii)
    df_pred = pd.DataFrame({
        "model":   model_id,
        "obs_idx": np.arange(n),
        "treated": usedata['treated'].values,
        "dy":      usedata['dy'].values,
        "pd":      pd_fit[:, 0],
    })
    for r in range(9):
        df_pred["yGd0_l%d" % (r + 1)] = yGd0x_fit[:, r]
        df_pred["dGx_l%d"  % (r + 1)] = dGx_fit[:, r]
    df_pred["yGd0_ss"] = yGd0_ss
    df_pred["dGx_ss"]  = dGx_ss
    preds_post.append(df_pred)

    # Trim propensity scores of 1 to .95 (per-learner AND short-stack)
    for r in range(9):
        trimmed[ii, r] = np.sum(dGx_fit[:, r] > 0.95)
        dGx_fit[dGx_fit[:, r] > 0.95, r] = 0.95
    dGx_ss[dGx_ss > 0.95] = 0.95

    # Calculate ATT and SE
    att_num = np.array([
        np.mean(((usedata['treated'].values - dGx_fit[:, r]) / (pd_fit[:, 0] * (1 - dGx_fit[:, r]))) *
                (usedata['dy'].values - yGd0x_fit[:, r]))
        for r in range(9)
    ])
    best_d = np.argmin(RMSE_d[ii, :])
    best_y = np.argmin(RMSE_y[ii, :])
    att_num_best = np.mean(((usedata['treated'].values - dGx_fit[:, best_d]) / (pd_fit[:, 0] * (1 - dGx_fit[:, best_d]))) *
                           (usedata['dy'].values - yGd0x_fit[:, best_y]))
    att_num_ss = np.mean(((usedata['treated'].values - dGx_ss) / (pd_fit[:, 0] * (1 - dGx_ss))) *
                        (usedata['dy'].values - yGd0_ss))
    att_den = np.mean(usedata['treated'].values / pd_fit[:, 0])
    att[ii, :9] = att_num / att_den
    att[ii, 9]  = att_num_best / att_den
    att[ii, 10] = att_num_ss  / att_den

    phihat = np.column_stack([
        ((usedata['treated'].values - dGx_fit[:, r]) / (pd_fit[:, 0] * (1 - dGx_fit[:, r]))) *
        (usedata['dy'].values - yGd0x_fit[:, r])
        for r in range(9)
    ])
    phihat_best = ((usedata['treated'].values - dGx_fit[:, best_d]) / (pd_fit[:, 0] * (1 - dGx_fit[:, best_d]))) * \
                  (usedata['dy'].values - yGd0x_fit[:, best_y])
    phihat_ss = ((usedata['treated'].values - dGx_ss) / (pd_fit[:, 0] * (1 - dGx_ss))) * \
                (usedata['dy'].values - yGd0_ss)
    phihat = np.column_stack([phihat, phihat_best, phihat_ss]) / att_den
    se_att[ii, :] = np.sqrt(np.mean(phihat ** 2, axis=0) / n)

In [6]:
# Remove columns for placebo test
drop_vars2 = ["lpop", "lavg_pay", "year", "G", "region"]
treat2_ = treat2.drop(columns=[col for col in drop_vars2 if col in treat2.columns])
treat2_['treated'] = 1

tdid02 = pd.merge(treat2_, treatB, on="id")
tdid02["dy"] = tdid02["lemp"] - tdid02["lemp.pre"]
tdid02 = tdid02.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in tdid02.columns])

cont2_ = cont2.drop(columns=[col for col in drop_vars2 if col in cont2.columns])
cont2_['treated'] = 0

cdid02 = pd.merge(cont2_, contB, on="id")
cdid02["dy"] = cdid02["lemp"] - cdid02["lemp.pre"]
cdid02 = cdid02.drop(columns=[col for col in ["id", "lemp", "lemp.pre"] if col in cdid02.columns])

In [7]:
# Prepare data for placebo year
tdata = tdid02.copy()
cdata = cdid02.copy()
usedata = pd.concat([tdata, cdata], ignore_index=True)

n = usedata.shape[0]
Kf = 5
kf = KFold(n_splits=Kf, shuffle=True, random_state=772023)

yGd0x_fit = np.zeros((n, 9))
dGx_fit = np.zeros((n, 9))
pd_fit = np.zeros((n, 1))

features = ['region', 'lemp.0', 'lpop.0', 'lavg_pay.0']
X = usedata[features].copy()
X = pd.get_dummies(X, columns=['region'], drop_first=True)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X)

for k, (train_idx, test_idx) in enumerate(kf.split(usedata)):
    ktrain = usedata.iloc[train_idx]
    ktest = usedata.iloc[test_idx]
    xtrain = X_poly[train_idx]
    xtest = X_poly[test_idx]
    ytrain = ktrain['dy'].values
    dtrain = ktrain['treated'].values

    # P(D=1)
    pd_fit[test_idx, 0] = dtrain.mean()

    # E[D|X]
    dGx_fit[test_idx, 0] = dtrain.mean()
    logit = LogisticRegression(solver='lbfgs', max_iter=1000)
    logit.fit(ktrain[features], dtrain)
    dGx_fit[test_idx, 1] = logit.predict_proba(ktest[features])[:, 1]
    logit2 = LogisticRegression(solver='lbfgs', max_iter=1000)
    logit2.fit(xtrain, dtrain)
    dGx_fit[test_idx, 2] = logit2.predict_proba(xtest)[:, 1]
    lasso = LassoCV(cv=5)
    lasso.fit(xtrain, dtrain)
    dGx_fit[test_idx, 3] = lasso.predict(xtest)
    ridge = RidgeCV(alphas=np.logspace(-6, 6, 13), cv=5)
    ridge.fit(xtrain, dtrain)
    dGx_fit[test_idx, 4] = ridge.predict(xtest)
    rf = RandomForestClassifier(n_estimators=1000, max_features=4, random_state=772023)
    rf.fit(ktrain[features], dtrain)
    dGx_fit[test_idx, 5] = rf.predict_proba(ktest[features])[:, 1]
    dt_deep = DecisionTreeClassifier(max_depth=15, min_samples_split=10, random_state=772023)
    dt_deep.fit(ktrain[features], dtrain)
    dGx_fit[test_idx, 6] = dt_deep.predict_proba(ktest[features])[:, 1]
    dt_shallow = DecisionTreeClassifier(max_depth=3, min_samples_split=10, random_state=772023)
    dt_shallow.fit(ktrain[features], dtrain)
    dGx_fit[test_idx, 7] = dt_shallow.predict_proba(ktest[features])[:, 1]
    dGx_fit[test_idx, 8] = dt_deep.predict_proba(ktest[features])[:, 1]

    # E[Y|D=0,X]
    ktrain0 = ktrain[ktrain['treated'] == 0]
    xtrain0 = X_poly[train_idx][ktrain['treated'] == 0]
    ytrain0 = ktrain0['dy'].values

    yGd0x_fit[test_idx, 0] = ytrain0.mean() if len(ytrain0) > 0 else 0
    lm = LinearRegression()
    lm.fit(ktrain0[features], ytrain0)
    yGd0x_fit[test_idx, 1] = lm.predict(ktest[features])
    lm2 = LinearRegression()
    lm2.fit(xtrain0, ytrain0)
    yGd0x_fit[test_idx, 2] = lm2.predict(xtest)
    lasso_y = LassoCV(cv=5)
    lasso_y.fit(xtrain0, ytrain0)
    yGd0x_fit[test_idx, 3] = lasso_y.predict(xtest)
    ridge_y = RidgeCV(alphas=np.logspace(-6, 6, 13), cv=5)
    ridge_y.fit(xtrain0, ytrain0)
    yGd0x_fit[test_idx, 4] = ridge_y.predict(xtest)
    rf_y = RandomForestRegressor(n_estimators=1000, max_features=4, random_state=772023)
    rf_y.fit(ktrain0[features], ytrain0)
    yGd0x_fit[test_idx, 5] = rf_y.predict(ktest[features])
    dt_deep_y = DecisionTreeRegressor(max_depth=15, min_samples_split=10, random_state=772023)
    dt_deep_y.fit(ktrain0[features], ytrain0)
    yGd0x_fit[test_idx, 6] = dt_deep_y.predict(ktest[features])
    dt_shallow_y = DecisionTreeRegressor(max_depth=3, min_samples_split=10, random_state=772023)
    dt_shallow_y.fit(ktrain0[features], ytrain0)
    yGd0x_fit[test_idx, 7] = dt_shallow_y.predict(ktest[features])
    yGd0x_fit[test_idx, 8] = dt_deep_y.predict(ktest[features])

# Calculate RMSE, ATT, SE, and trimming for placebo year
RMSE_dP = np.sqrt(np.mean((usedata['treated'].values.reshape(-1, 1) - dGx_fit) ** 2, axis=0))
RMSE_yP = np.sqrt(np.mean((usedata['dy'].values[usedata['treated'] == 0].reshape(-1, 1) -
                           yGd0x_fit[usedata['treated'] == 0, :]) ** 2, axis=0))

# Short-stack on un-trimmed cross-fit predictions.
d0_mask = usedata['treated'].values == 0
w_y_P = nnls_normalize(usedata['dy'].values[d0_mask],
                       yGd0x_fit[d0_mask, :])
w_d_P = nnls_normalize(usedata['treated'].values, dGx_fit)
yGd0_ss_P = yGd0x_fit @ w_y_P
dGx_ss_P  = dGx_fit  @ w_d_P

# Snapshot raw predictions for downstream diagnostics.
df_pred_P = pd.DataFrame({
    "model":   "m02",
    "obs_idx": np.arange(n),
    "treated": usedata['treated'].values,
    "dy":      usedata['dy'].values,
    "pd":      pd_fit[:, 0],
})
for r in range(9):
    df_pred_P["yGd0_l%d" % (r + 1)] = yGd0x_fit[:, r]
    df_pred_P["dGx_l%d"  % (r + 1)] = dGx_fit[:, r]
df_pred_P["yGd0_ss"] = yGd0_ss_P
df_pred_P["dGx_ss"]  = dGx_ss_P

trimmedP = np.array([np.sum(dGx_fit[:, r] > 0.95) for r in range(9)])
for r in range(9):
    dGx_fit[dGx_fit[:, r] > 0.95, r] = 0.95
dGx_ss_P[dGx_ss_P > 0.95] = 0.95

att_numP = np.array([
    np.mean(((usedata['treated'].values - dGx_fit[:, r]) / (pd_fit[:, 0] * (1 - dGx_fit[:, r]))) *
            (usedata['dy'].values - yGd0x_fit[:, r]))
    for r in range(9)
])
best_dP = np.argmin(RMSE_dP)
best_yP = np.argmin(RMSE_yP)
att_num_bestP = np.mean(((usedata['treated'].values - dGx_fit[:, best_dP]) / (pd_fit[:, 0] * (1 - dGx_fit[:, best_dP]))) *
                        (usedata['dy'].values - yGd0x_fit[:, best_yP]))
att_num_ssP = np.mean(((usedata['treated'].values - dGx_ss_P) / (pd_fit[:, 0] * (1 - dGx_ss_P))) *
                      (usedata['dy'].values - yGd0_ss_P))
att_denP = np.mean(usedata['treated'].values / pd_fit[:, 0])
attP = np.zeros(11)
attP[:9] = att_numP / att_denP
attP[9]  = att_num_bestP / att_denP
attP[10] = att_num_ssP  / att_denP

phihatP = np.column_stack([
    ((usedata['treated'].values - dGx_fit[:, r]) / (pd_fit[:, 0] * (1 - dGx_fit[:, r]))) *
    (usedata['dy'].values - yGd0x_fit[:, r])
    for r in range(9)
])
phihat_bestP = ((usedata['treated'].values - dGx_fit[:, best_dP]) / (pd_fit[:, 0] * (1 - dGx_fit[:, best_dP]))) * \
               (usedata['dy'].values - yGd0x_fit[:, best_yP])
phihat_ssP = ((usedata['treated'].values - dGx_ss_P) / (pd_fit[:, 0] * (1 - dGx_ss_P))) * \
             (usedata['dy'].values - yGd0_ss_P)
phihatP = np.column_stack([phihatP, phihat_bestP, phihat_ssP]) / att_denP
se_attP = np.sqrt(np.mean(phihatP ** 2, axis=0) / n)

In [8]:
# Display RMSE for Y
table1y = pd.DataFrame(RMSE_y.T, columns=["2004", "2005", "2006", "2007"],
                       index=["No Controls", "Basic", "Expansion", "Lasso (CV)", "Ridge (CV)",
                              "Random Forest", "Deep Tree", "Shallow Tree", "Tree (CV)"])
print("RMSE for Y:")
print(table1y.round(4))

# Display RMSE for D
table1d = pd.DataFrame(RMSE_d.T, columns=["2004", "2005", "2006", "2007"],
                       index=["No Controls", "Basic", "Expansion", "Lasso (CV)", "Ridge (CV)",
                              "Random Forest", "Deep Tree", "Shallow Tree", "Tree (CV)"])
print("\nRMSE for D:")
print(table1d.round(4))

# Display ATT and SE
att_se_rows = []
for i in range(11):
    att_se_rows.append(att[:, i])
    att_se_rows.append(se_att[:, i])

row_names = ["No Controls", "s.e.", "Basic", "s.e.", "Expansion", "s.e.", "Lasso (CV)", "s.e.",
             "Ridge (CV)", "s.e.", "Random Forest", "s.e.", "Deep Tree", "s.e.", "Shallow Tree", "s.e.",
             "Tree (CV)", "s.e.", "Best", "s.e.", "Stack", "s.e."]

table2 = pd.DataFrame(np.vstack(att_se_rows).T, columns=row_names, index=["2004", "2005", "2006", "2007"])
print("\nATT and SE:")
print(table2.round(3))

# Short-stack weights per year (rows = years)
learner_names = ["No Controls", "Basic", "Expansion", "Lasso (CV)", "Ridge (CV)",
                 "Random Forest", "Deep Tree", "Shallow Tree", "Tree (CV)"]
print("\nShort-stack weights for Y|X,D=0:")
print(pd.DataFrame(stack_w_y_post, columns=learner_names,
                   index=["2004", "2005", "2006", "2007"]).round(3))
print("\nShort-stack weights for D|X:")
print(pd.DataFrame(stack_w_d_post, columns=learner_names,
                   index=["2004", "2005", "2006", "2007"]).round(3))

# Display number of trimmed observations
print("\nNumber of trimmed observations (propensity > 0.95):")
print(pd.DataFrame(trimmed, columns=learner_names,
                   index=["2004", "2005", "2006", "2007"]))

# Display placebo (pre-trend) results
print("\nPlacebo (pre-trend, 2002) results:")
print("RMSE Y:", RMSE_yP)
print("RMSE D:", RMSE_dP)
print("ATT:", attP)
print("SE:", se_attP)
print("Trimmed:", trimmedP)
print("Short-stack wY (2002):", np.round(w_y_P, 3))
print("Short-stack wD (2002):", np.round(w_d_P, 3))

RMSE for Y:
                 2004    2005    2006    2007
No Controls    0.1634  0.1882  0.2235  0.2301
Basic          0.1633  0.1854  0.2186  0.2215
Expansion      0.1686  0.1898  0.2235  0.2267
Lasso (CV)     0.1634  0.1848  0.2176  0.2219
Ridge (CV)     0.1630  0.1847  0.2174  0.2215
Random Forest  0.1721  0.2002  0.2369  0.2388
Deep Tree      0.1957  0.2266  0.2704  0.2769
Shallow Tree   0.1715  0.1920  0.2199  0.2250
Tree (CV)      0.1957  0.2266  0.2704  0.2769

RMSE for D:
                 2004    2005    2006    2007
No Controls    0.1982  0.2006  0.2112  0.2503
Basic          0.1981  0.2005  0.2111  0.2192
Expansion      0.1884  0.1900  0.1985  0.2158
Lasso (CV)     0.1982  0.2006  0.2112  0.2503
Ridge (CV)     0.1891  0.1914  0.2007  0.2145
Random Forest  0.2010  0.2020  0.2119  0.2346
Deep Tree      0.2390  0.2463  0.2503  0.2756
Shallow Tree   0.1916  0.1931  0.2038  0.2250
Tree (CV)      0.2390  0.2463  0.2503  0.2756

ATT and SE:
      No Controls   s.e.  Basic   s.e.  Ex

In [9]:
# Save raw cross-fit predictions for downstream diagnostics
# (Python code/mwDiD_diagnostics.py reads this).
#
# Long-form: one row per observation, with model_id distinguishing
# m02 (placebo) and m04..m07 (dynamic ATETs). Mirrors the .dta saved by
# Stata Code/mwDiD.do.

preds_all = pd.concat([df_pred_P, *preds_post], ignore_index=True)

out_path = os.path.join("..", "Data", "mwDiD_python.parquet")
preds_all.to_parquet(out_path, index=False)
print(f"Wrote {len(preds_all):,} rows of cross-fit predictions to {out_path}")

Wrote 11,218 rows of cross-fit predictions to ..\Data\mwDiD_python.parquet
